# Food Price CPI — 18-Month Forecast Experiment (CFPR Replica)

This notebook runs the 18-month food CPI forecasting experiment, replicating the
evaluation structure of Canada's Food Price Report (CFPR).

**Task:** Forecast Canadian food CPI index values 18 months ahead across 9 food categories.  
**Predictors compared:**
- `LastValuePredictor` — naive baseline (performance floor)
- `DartsAutoARIMAPredictor` — univariate AutoARIMA (Darts `AutoARIMA` has no exogenous covariate support in this stack)

**Primary metric:** CRPS (lower is better).  
**Supplementary metric:** MAPE on median forecast (computed post-hoc).  
**Reporting:** Year-over-year % change derived from index forecasts.

## Sections
1. Setup: register data and load specs
2. Run multi-target backtest
3. CRPS comparison table (all predictors × all categories)
4. MAPE comparison (post-hoc, median forecasts)
5. Run multi-target eval (protected window)
6. Forecast report: year-over-year % change

## Prerequisites

```bash
uv run python scripts/fetch_cpi.py
uv run python scripts/fetch_fred.py   # requires FRED_API_KEY in .env
```

In [1]:
import sys
from pathlib import Path

import yaml

# Notebook lives at implementations/experiments/food_price_forecasting/ — 3 levels deep.
repo_root = Path().resolve().parents[2]
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

## 1. Setup: Register Data and Load Specs

In [2]:
import os
from datetime import datetime, timezone

import numpy as np
import pandas as pd
from dotenv import load_dotenv

from aieng.forecasting.data import DataService, SeriesMetadata
from aieng.forecasting.data.adapters import FREDAdapter, StatCanAdapter
from aieng.forecasting.evaluation import (
    MultiTargetBacktestSpec,
    MultiTargetEvalSpec,
    multi_backtest,
    multi_evaluate,
)
from aieng.forecasting.evaluation.eval import EvalTracker
from methods.darts_arima import DartsAutoARIMAPredictor
from methods.naive import LastValuePredictor

load_dotenv(repo_root / ".env")

CPI_TABLE_ID = "18-10-0004-11"
CACHE_DIR = repo_root / "data" / "statcan"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

svc = DataService()

# --- StatCan food CPI series ---
FOOD_CPI_SERIES = [
    ("cpi_food_canada", "Food"),
    ("cpi_bakery_cereal_canada", "Bakery and cereal products (excluding baby food)"),
    ("cpi_dairy_eggs_canada", "Dairy products and eggs"),
    ("cpi_fish_seafood_canada", "Fish, seafood and other marine products"),
    ("cpi_restaurants_canada", "Food purchased from restaurants"),
    ("cpi_fruit_preparations_nuts_canada", "Fruit, fruit preparations and nuts"),
    ("cpi_meat_canada", "Meat"),
    ("cpi_other_food_nonalcoholic_canada", "Other food products and non-alcoholic beverages"),
    ("cpi_vegetables_preparations_canada", "Vegetables and vegetable preparations"),
]

for series_id, product_group in FOOD_CPI_SERIES:
    svc.register(
        series_id,
        StatCanAdapter(
            table_id=CPI_TABLE_ID,
            member_filter={"GEO": "Canada", "Products and product groups": product_group},
            cache_dir=CACHE_DIR,
        ),
        SeriesMetadata(
            series_id=series_id,
            description=f"CPI {product_group}, Canada (2002=100)",
            source="StatCan",
            units="Index 2002=100",
            frequency="MS",
            table_id=CPI_TABLE_ID,
        ),
    )

# --- FRED covariate series ---
FRED_SERIES = [
    ("fred_us_cpi_food_at_home", "CPIFABSL", "US CPI: Food at Home (1982-84=100)", "Index 1982-84=100"),
    ("fred_us_cpi_meats_poultry_fish_eggs", "CUSR0000SAF112", "US CPI: Meats, Poultry, Fish, Eggs", "Index 1982-84=100"),
    ("fred_us_cpi_fruits_vegetables", "CUSR0000SAF113", "US CPI: Fruits & Vegetables", "Index 1982-84=100"),
    ("fred_canada_10yr_bond_yield", "IRLTLT01CAM156N", "Canada 10yr Bond Yield (%)", "Percent"),
    ("fred_canada_us_exchange_rate", "EXCAUS", "Canada/US Exchange Rate (CAD/USD)", "CAD per USD"),
    ("fred_sp100_volatility_vxo", "VXOCLS", "S&P 100 Volatility Index (VXO)", "Index"),
    ("fred_wilshire_5000", "WILL5000IND", "Wilshire 5000 Total Market Index", "Index"),
]

fred_available = []
for series_id, fred_id, description, units in FRED_SERIES:
    try:
        svc.register(
            series_id,
            FREDAdapter(fred_id),
            SeriesMetadata(
                series_id=series_id,
                description=description,
                source=f"FRED ({fred_id})",
                units=units,
                frequency="MS",
            ),
        )
        fred_available.append(series_id)
    except Exception as e:
        print(f"  [WARN] Skipping {series_id}: {e}")

print(f"\nRegistered: {len(FOOD_CPI_SERIES)} food CPI + {len(fred_available)} FRED series.")

  [WARN] Skipping fred_wilshire_5000: Failed to fetch FRED series 'WILL5000IND': Bad Request.  The series does not exist.

Registered: 9 food CPI + 6 FRED series.


In [3]:
# Load backtest spec
spec_path = repo_root / "reference_specs" / "food_cpi" / "food_cpi_18m_backtest.yaml"
with spec_path.open() as f:
    backtest_spec = MultiTargetBacktestSpec.model_validate(yaml.safe_load(f))

print(f"Backtest spec: {len(backtest_spec.tasks)} tasks")
print(f"Window: {backtest_spec.start.date()} → {backtest_spec.end.date()}")
print(f"Stride: {backtest_spec.stride} months, warmup: {backtest_spec.warmup} obs")
print(f"Candidate origins: {len(backtest_spec.specs()[0].origins())}")

Backtest spec: 9 tasks
Window: 2000-01-01 → 2026-01-01
Stride: 6 months, warmup: 24 obs
Candidate origins: 53


In [4]:
# Define predictors
predictors = [
    LastValuePredictor(),
    DartsAutoARIMAPredictor(num_samples=500),
]

for p in predictors:
    print(f"  {p.predictor_id}")

  last_value_naive
  darts_autoarima


## 2. Run Multi-Target Backtest

Runs each predictor across all 9 food categories.  AutoARIMA fits once per origin per category — expect several minutes per predictor.

In [ ]:
import time

all_results = {}  # predictor_id → dict[task_id, BacktestResult]

for predictor in predictors:
    print(f"Running backtest: {predictor.predictor_id} ...")
    t0 = time.time()
    results = multi_backtest(predictor=predictor, spec=backtest_spec, data_service=svc)
    elapsed = time.time() - t0
    mean_crps = np.mean([r.mean_crps for r in results.values()])
    print(f"  Done in {elapsed:.0f}s — mean CRPS across categories: {mean_crps:.4f}")
    all_results[predictor.predictor_id] = results

Running backtest: last_value_naive ...
  Done in 3s — mean CRPS across categories: 7.0102
Running backtest: darts_autoarima ...


Support for PyTorch based likelihood models not available. To enable them, install "darts[torch]" or "darts[all]" (with pip); or "u8darts-torch" or "u8darts-all" (with conda).
Support for Torch based models not available. To enable them, install "darts[torch]" or "darts[all]" (with pip); or "u8darts-torch" or "u8darts-all" (with conda).
/home/ethanjackson/agentic-forecasting/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 3. CRPS Comparison Table

In [ ]:
CATEGORY_LABELS = {
    "food_cpi_overall_18m": "Overall Food",
    "food_cpi_bakery_cereal_18m": "Bakery & Cereal",
    "food_cpi_dairy_eggs_18m": "Dairy & Eggs",
    "food_cpi_fish_seafood_18m": "Fish & Seafood",
    "food_cpi_restaurants_18m": "Restaurants",
    "food_cpi_fruit_preparations_nuts_18m": "Fruit & Nuts",
    "food_cpi_meat_18m": "Meat",
    "food_cpi_other_food_18m": "Other Food",
    "food_cpi_vegetables_18m": "Vegetables",
}

crps_data = {}
for predictor_id, task_results in all_results.items():
    crps_data[predictor_id] = {
        CATEGORY_LABELS.get(tid, tid): round(r.mean_crps, 4)
        for tid, r in task_results.items()
    }

crps_df = pd.DataFrame(crps_data)
crps_df.loc["Mean"] = crps_df.mean().round(4)
crps_df

In [ ]:
# CRPS reduction vs. naive baseline (positive = improvement)
if "last_value_naive" in crps_df.columns:
    for col in crps_df.columns:
        if col != "last_value_naive":
            crps_df[f"reduction_vs_naive_{col}"] = (
                (crps_df["last_value_naive"] - crps_df[col]) / crps_df["last_value_naive"] * 100
            ).round(1)

crps_df.style.format("{:.4f}", subset=[c for c in crps_df.columns if not c.startswith("reduction")]).format(
    "{:.1f}%", subset=[c for c in crps_df.columns if c.startswith("reduction")]
)

## 4. MAPE Comparison (Supplementary)

MAPE is computed post-hoc from the median forecast (`point_forecast`) for each origin.

In [ ]:
def compute_mape(task_results: dict) -> dict[str, float]:
    """Compute mean absolute percentage error from BacktestResult predictions.

    Uses the resolution logic: the actual value is inferred from the
    BacktestResult's score, but for MAPE we need the actual value directly.
    We re-query the data service.
    """
    as_of_now = datetime.now(tz=timezone.utc).replace(tzinfo=None)
    mapes = {}
    for task_id, result in task_results.items():
        target_series_id = result.spec.task.target_series_id
        full_series = svc.get_series(target_series_id, as_of=as_of_now)
        full_series["timestamp"] = pd.to_datetime(full_series["timestamp"])
        full_series = full_series.set_index("timestamp")["value"]

        ape_list = []
        for pred in result.predictions:
            ts = pd.Timestamp(pred.forecast_date)
            if ts in full_series.index:
                actual = full_series[ts]
                median_forecast = pred.payload.point_forecast
                if actual != 0:
                    ape_list.append(abs(median_forecast - actual) / abs(actual) * 100)

        mapes[CATEGORY_LABELS.get(task_id, task_id)] = round(float(np.mean(ape_list)), 3) if ape_list else float("nan")
    return mapes


mape_data = {}
for predictor_id, task_results in all_results.items():
    mape_data[predictor_id] = compute_mape(task_results)

mape_df = pd.DataFrame(mape_data)
mape_df.loc["Mean"] = mape_df.mean().round(3)
mape_df

## 5. Multi-Target Eval (Protected Window)

One call to `multi_evaluate` counts as one budget run for all 9 categories.
Use sparingly — `max_runs=5` per tracker file.

In [ ]:
eval_spec_path = repo_root / "reference_specs" / "food_cpi" / "food_cpi_18m_eval.yaml"
with eval_spec_path.open() as f:
    eval_spec = MultiTargetEvalSpec.model_validate(yaml.safe_load(f))

print(f"Eval spec: {eval_spec.spec_id}")
print(f"Window: {eval_spec.start.date()} → {eval_spec.end.date()}")
print(f"Max runs: {eval_spec.max_runs}")

tracker = EvalTracker(Path("eval_runs_18m.yaml"))
print(f"Runs used so far: {tracker.runs_for(eval_spec.spec_id)} / {eval_spec.max_runs}")

In [ ]:
# Uncomment to run eval — costs one budget run.
# Choose the predictor you want to evaluate on the protected window.

# best_predictor = DartsAutoARIMAPredictor(num_samples=500)
# eval_results = multi_evaluate(
#     predictor=best_predictor,
#     spec=eval_spec,
#     data_service=svc,
#     tracker=tracker,
# )

# eval_crps = {CATEGORY_LABELS.get(tid, tid): round(r.mean_crps, 4) for tid, r in eval_results.items()}
# eval_series = pd.Series(eval_crps, name="eval_mean_crps")
# print(f"Eval mean CRPS across categories: {eval_series.mean():.4f}")
# print(eval_series)
print("Eval cell ready — uncomment to consume one budget run.")

## 6. Forecast Report: Year-over-Year % Change

The CFPR reports food price predictions as year-over-year % change ranges.
Here we derive YoY % from the index forecast.

**Method:** `yoy_pct = (forecast_index - actual_12m_before_forecast_date) / actual_12m_before_forecast_date * 100`

**Uncertainty range:** CRPS-weighted mean MAPE from the backtest is used as an indicative ±range. A better calibration approach would use the predictive quantiles directly.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

as_of_now = datetime.now(tz=timezone.utc).replace(tzinfo=None)

# Show the most recent (last) prediction for each category from the best predictor.
# The "best" predictor here is the one with the lowest mean CRPS.
best_pid = crps_df.drop("Mean").mean().idxmin()
best_task_results = all_results[best_pid]

print(f"Best predictor by mean CRPS: {best_pid}")
print()
print(f"{'Category':<40} {'Forecast Date':<16} {'Median Index':<16} {'YoY %':>8} {'90% PI':>20}")
print("-" * 105)

for task_id, result in best_task_results.items():
    if not result.predictions:
        continue
    last_pred = result.predictions[-1]
    target_sid = result.spec.task.target_series_id
    full_series = svc.get_series(target_sid, as_of=as_of_now)
    full_series["timestamp"] = pd.to_datetime(full_series["timestamp"])
    full_series = full_series.set_index("timestamp")["value"]

    # Lookup value 12 months before the forecast date.
    ref_ts = pd.Timestamp(last_pred.forecast_date) - pd.DateOffset(months=12)
    if ref_ts in full_series.index:
        ref_value = full_series[ref_ts]
        median_idx = last_pred.payload.point_forecast
        yoy = (median_idx - ref_value) / ref_value * 100

        q10 = last_pred.payload.quantiles.get(0.10, median_idx)
        q90 = last_pred.payload.quantiles.get(0.90, median_idx)
        yoy_lo = (q10 - ref_value) / ref_value * 100
        yoy_hi = (q90 - ref_value) / ref_value * 100

        label = CATEGORY_LABELS.get(task_id, task_id)
        fd = last_pred.forecast_date.strftime("%Y-%m")
        print(f"{label:<40} {fd:<16} {median_idx:>12.1f}   {yoy:>+7.1f}%  [{yoy_lo:+.1f}%, {yoy_hi:+.1f}%]")

In [ ]:
# Visualise backtest predictions for Overall Food with 80% prediction interval.
overall_task_id = "food_cpi_overall_18m"
result = best_task_results[overall_task_id]
target_sid = result.spec.task.target_series_id

obs_df = svc.get_series(target_sid, as_of=as_of_now)
obs_df["timestamp"] = pd.to_datetime(obs_df["timestamp"])

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 9), gridspec_kw={"height_ratios": [3, 1]})

# Observations — last 15 years
cutoff = pd.Timestamp("2010-01-01")
obs_recent = obs_df[obs_df["timestamp"] >= cutoff]
ax1.plot(obs_recent["timestamp"], obs_recent["value"], color="#374151", linewidth=1.5, label="Observed", zorder=3)

# Forecasts: median + 80% PI
pred_dates = [pd.Timestamp(p.forecast_date) for p in result.predictions]
medians = [p.payload.point_forecast for p in result.predictions]
q10s = [p.payload.quantiles.get(0.10, p.payload.point_forecast) for p in result.predictions]
q90s = [p.payload.quantiles.get(0.90, p.payload.point_forecast) for p in result.predictions]

ax1.scatter(pred_dates, medians, color="#2563eb", s=25, zorder=4, label=f"{best_pid} (median)")
ax1.fill_between(pred_dates, q10s, q90s, alpha=0.25, color="#2563eb", label="80% PI")

ax1.set_title("Canada Food CPI — Overall Food: 18-Month Ahead Backtest", fontsize=12, fontweight="bold")
ax1.set_ylabel("Index (2002=100)")
ax1.legend()
ax1.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax1.xaxis.set_major_locator(mdates.YearLocator(2))

# Per-origin CRPS
origin_dates = [pd.Timestamp(p.as_of) for p in result.predictions]
ax2.bar(origin_dates, result.scores, width=60, color="#7c3aed", alpha=0.7, label="CRPS")
ax2.axhline(result.mean_crps, color="#dc2626", linewidth=1.5, linestyle="--", label=f"Mean CRPS: {result.mean_crps:.3f}")
ax2.set_ylabel("CRPS")
ax2.set_xlabel("Forecast origin")
ax2.legend(fontsize=9)
ax2.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax2.xaxis.set_major_locator(mdates.YearLocator(2))

fig.tight_layout()
plt.show()